In [1]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path

# Create datasets for TIMIT MTurk task


## What this will create:

10 versions of the same dataset that differ only by the excerpts included via random sampling.

For each set: 
* Create 400 trials with the following conditions: 
    * Distractor conditions 
      * 1-distractor
      * 2-distractor
      * 4-distractor
      * Speech-shaped noise
    * Signal to noise ratios:
      * -15 dB
      * -5 dB
      * 0 dB
      * +10 dB
    * Limit target words to not occur more than twice
    * Space out repeated target words across trials to minimize priming effects 
    * Balance distractor sex pairings 
        * eg 50-50 for 1-distractor; 25-25-25-25 for 4-distractor
    
All sets generated with random seed == 0 



### What's included in each final dataframe

Output 10 dataframes each with the following data.


##### Note: rows are trials and columns are trial stimuli and metadata.
original columns included: 

       'word', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'
      
with addition of:

        '_original_timit_index', '_original_cue_timit_index', 'cue_signal', 'cue_word', 'distractor', 
        '_original_distractor_timit_index', 'distractor_words', 'distractor_speakers', 'distractor_condition' 
        
where `_original_<type>_ix` maps audio sources to their original row in  `/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl`
 
 
## Need to include catch trials in dataframes 

Existing files were not named, so it's easier to draw from timit that match wavs to unique talkers

## Load timit dataset

In [2]:
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'

timit_excerpts = pd.read_pickle(path)


In [3]:
## Set random seed for exp generation (init to 0) 

np.random.seed(0)
random.seed(0)

In [4]:
# Check that we can get sex balance 

In [5]:
timit_excerpts.speaker_sex.value_counts()

m    7117
f    3192
Name: speaker_sex, dtype: int64

In [6]:
timit_excerpts[timit_excerpts.word_int != 0].speaker_sex.value_counts()

m    620
f    282
Name: speaker_sex, dtype: int64

### Normalize timit audio before cutting

In [7]:
all_signals = np.stack(timit_excerpts.signal)

In [8]:
all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


In [9]:
per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


In [10]:
all_signals = all_signals * 0.02 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

In [11]:
timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

## Segment into target and cue portions of TIMIT 

Targets and distractors will be drawn from target_timit, while cues will come from cue_timit. This way, cues will never contain target words, but distractors will. 

Distractors will be drawn from the complimentary set of targets for a given experiment, so no excerpt will be both a target and distractor in a given experiment. 

In [12]:
timit_excerpts.sentence_id.value_counts() # see most frequent sentences 

1       1479
2       1373
174       24
371       23
313       22
        ... 
843        1
1334       1
1222       1
1906       1
563        1
Name: sentence_id, Length: 1976, dtype: int64

In [13]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

cue_timit = timit_excerpts[timit_excerpts.word_int == 0]

# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]

# Cut targets to minimize word repetition 
valid_words = target_timit.word.value_counts()[target_timit.word.value_counts() < 9].index

# get training stim for start of experiment 
training_timit = target_timit[~target_timit.word.isin(valid_words)]

target_timit = target_timit[target_timit.word.isin(valid_words)]


In [14]:
cue_timit.shape

(9407, 13)

In [15]:
# Save df of catch trials & demo cue selection logic

# pull 10 training examples 

train_targets = target_timit.groupby('speaker_sex').sample(n=10).copy() # replace=False is default
train_targets.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

train_targets.index = train_targets.index.set_names(['_original_timit_index']) # add original index mapping
train_targets = train_targets.reset_index() # Reset for 0:10 ixs 

# get set of distractors given targets
bg_ixs = target_timit[["speaker", 'sentence_id','word']].merge(train_targets[['speaker', 'sentence_id','word']],
                                how = 'outer' ,indicator=True).loc[lambda x : x['_merge']=='left_only'].index
bg_timit = target_timit.iloc[bg_ixs]
bg_timit.index = bg_timit.index.set_names(['_original_timit_index']) # keep original ixs 
bg_timit = bg_timit.reset_index()

print(f"{len(train_targets.word.unique())} unique words ")
print(f"{train_targets.word.value_counts().max()} max frequency ")

# Sample cues 
## To sample cues, get list of talkers in target set 

talkers = train_targets.speaker.unique()

# sample 1 cue per excerpt from viable cues 
samples_per_talker = {talker:count for talker,count in train_targets.speaker.value_counts().items()}
viables_cues = cue_timit[cue_timit.speaker.isin(talkers)]

cues = viables_cues.groupby('speaker').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='speaker', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'_original_cue_timit_index'}, inplace=True)

train_targets.sort_values(by='speaker', inplace=True)
train_targets.reset_index(inplace=True, drop=True)


cues.sort_values(by='speaker', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = cues[['signal', 'word', '_original_cue_timit_index', 'speaker']]
# Combine as experiment dataframe
training_df = train_targets.join(cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']])
assert (training_df['speaker'] == training_df['cue_speaker']).all(), "Cue and Target talkers don't match!"




20 unique words 
1 max frequency 


In [18]:
training_df.head()

,_original_timit_index,word,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,signal,word_int,cue_signal,cue_word,_original_cue_timit_index,cue_speaker
0,8572,needed,test-dr4-fcft0-sx368,fcft0,20000,40000,f,sx,368,dr4,"[0.00011094248105231591, 0.0001679826731135585...",450,"[0.04792072963271731, 0.03598703685527364, 0.0...",oily,8563,fcft0
1,7679,expected,test-dr2-fcmr0-sx205,fcmr0,20000,40000,f,sx,205,dr2,"[-0.0005790408008140293, -0.001001866541698006...",253,"[-0.036377900099321694, -0.042766877007107414,...",up,7675,fcmr0
2,5935,situation,train-dr7-fcrz0-si2053,fcrz0,20000,40000,f,si,2053,dr7,"[0.008558774063336952, 0.0020222148789102706, ...",654,"[0.00358540716518209, 0.00266543354898426, 0.0...",an,5931,fcrz0
3,4305,music,train-dr5-fdmy0-sx297,fdmy0,20000,40000,f,sx,297,dr5,"[-0.045337446458735754, -0.04936817491795092, ...",441,"[-0.0003085411541804638, -6.263832622340351e-0...",pair,4299,fdmy0
4,66,light,train-dr1-fetb0-sx248,fetb0,20000,40000,f,sx,248,dr1,"[6.136765817866252e-05, 2.8603644386108485e-05...",393,"[0.00024669986441104385, 0.0003374186407384911...",carry,57,fetb0


In [19]:
training_df.shape

(20, 16)

In [20]:
out_name = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/attn_task_training_excerpts.pdpkl')

training_df.to_pickle(out_name)

In [21]:
 # we lose exact gener balance if we go below 9 words 7yu
valid_words = target_timit.word.value_counts()[target_timit.word.value_counts() < 9].index
target_timit[target_timit.word.isin(valid_words)].speaker_sex.value_counts()

m    505
f    223
Name: speaker_sex, dtype: int64

In [22]:
target_timit.shape

(728, 13)

In [21]:
timit_excerpts.columns

Index(['word', '_full_dataset_index', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'],
      dtype='object')

##  Make Dataframes


###  Define functions:


#### In final df, we will save trial stimuli mixture and cue signals and labels, with indices to original signal sources so they can be reconstructed. This will help keep the files light 

___ 

Define some useful functions for mixing audio

In [22]:

next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.02, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor
    

In [23]:
## Precompute spectrum of all signals for speech shaped noise 

all_foregrounds = np.stack(target_timit['signal'])
n_x = all_foregrounds.shape[-1]
n_fft = next_pow_2(n_x)
X = np.fft.rfft(all_foregrounds, next_pow_2(n_fft))
X = X.mean(0)

def generate_speech_shaped_noise(X=X):
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    ssn = noise[:n_x]
    return ssn

## Make Dataframes 

### Setup experimental conditions 


In [24]:
## Make trial conditions 
import itertools

n_male = n_female = 200 # force balance of foregrounds 
n_datasets = 10

# setup 4x4 design
snrs = [-6, -3, 0, 3] # 10/20/2022 Changed from -15, -5, 0, 10
backgrounds = [1,2,4,'ssn']

condition_pairs = list(itertools.product(snrs, backgrounds))

n_trials = 400 
n_catch_trials = 12 
n_conditions = len(condition_pairs )

trials_per_condition = n_trials // n_conditions

distractor_types = ['m', 'f']

two_talker_conds = list(itertools.combinations_with_replacement(distractor_types,2))
four_talker_conds = list(itertools.combinations_with_replacement(distractor_types,4))


file_out_dir = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/')

In [25]:
# define function for modifying cue levels 

    

def rove_cue(clean, snr, mixture_scale_factor):
    # clean is cue, noise is mixture here 
    # get ratio in rms 
#     snr = np.random.uniform(low=-20, high=0)
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    # scale factor for setting cue to desired SNR 
    # put ratio in units of second sound 
    # Blend signals 
    clean = clean * rms_ratio
    # scale by factor applied to target/distractor mixture
    clean = clean * mixture_scale_factor
#     mixture = clean + noise[:len(clean)]
    return snr, clean 


In [26]:
for set_ix in tqdm(range(n_datasets)):
    # Randomly generate experimental condition list
    all_conditions = []
    for condition_pair in condition_pairs:
        snr, condition = condition_pair
        for ix in range(trials_per_condition):
            if condition == 1:
                distractor = np.random.choice(distractor_types)
            elif condition == 2:
                type_ix = np.random.choice(len(two_talker_conds))
                distractor = two_talker_conds[type_ix]
            elif condition == 4:
                type_ix = np.random.choice(len(four_talker_conds))
                distractor = four_talker_conds[type_ix]
            else:
                distractor = 'ssn'

            all_conditions.append({'snr':snr,
                                  'condition':condition,
                                  'distractor':distractor})

    # shuffle condition order inplace
    random.shuffle(all_conditions)
    
    # sample our talkers for one set 
    set_targets = target_timit.groupby('speaker_sex').sample(n=n_female).copy() # replace=False is default
    set_targets.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    set_targets.index = set_targets.index.set_names(['_original_timit_index']) # add original index mapping
    set_targets = set_targets.reset_index() # Reset for 0:399 ixs 
    
    # get set of distractors given targets
    bg_ixs = target_timit[["speaker", 'sentence_id','word']].merge(set_targets[['speaker', 'sentence_id','word']],
                                    how = 'outer' ,indicator=True).loc[lambda x : x['_merge']=='left_only'].index
    bg_timit = target_timit.iloc[bg_ixs]
    bg_timit.index = bg_timit.index.set_names(['_original_timit_index']) # keep original ixs 
    bg_timit = bg_timit.reset_index()
    
    print(f"{len(set_targets.word.unique())} unique words in dataset {set_ix}")
    print(f"{set_targets.word.value_counts().max()} max frequency in dataset {set_ix}")
    
    # Sample cues 
    ## To sample cues, get list of talkers in target set 

    talkers = set_targets.speaker.unique()

    # sample 1 cue per excerpt from viable cues 
    samples_per_talker = {talker:count for talker,count in set_targets.speaker.value_counts().items()}
    viables_cues = cue_timit[cue_timit.speaker.isin(talkers)]

    cues = viables_cues.groupby('speaker').apply(lambda group: group.sample(samples_per_talker[group.name]))
    cues.drop(columns='speaker', inplace=True)
    cues = cues.reset_index()
    cues.rename(columns={'level_1':'_original_cue_timit_index'}, inplace=True)
    
    set_targets.sort_values(by='speaker', inplace=True)
    set_targets.reset_index(inplace=True, drop=True)
    
    ### Merge cues with foregrounds  
    cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = cues[['signal', 'word', '_original_cue_timit_index', 'speaker']]
    # Combine as experiment dataframe
    exp_df = set_targets.join(cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']])
    assert (exp_df['speaker'] == exp_df['cue_speaker']).all(), "Cue and Target talkers don't match!"
    
    # Generate trial by trial stim    
    trial_data = {'mixture_signal':[],
            'distractor_signal':[],
            '_original_distractor_timit_indices':[],
            'distractor_words':[],
            'distractor_speakers':[],
            'distractor_conditions':[],
            'distractor_sex':[],
            'snrs':[]}
    cue_signals = []
    cue_snrs = []

    # Loop over trials to make mixtures 
    for i in tqdm(range(n_trials), leave=False):
        # get signal
        signal = exp_df['signal'][i]
        speaker = exp_df['speaker'][i]
        valid_distractors = bg_timit[bg_timit['speaker'] != speaker].reset_index(drop=True) # reset so ixing works
        # get condition 
        snr, condition, distractors = all_conditions[i].values()
        # Get data based on condition
        if condition == 'ssn':
            noise = generate_speech_shaped_noise()
            indices = 'ssn'
            words = 'ssn'
            speakers = 'ssn'

        else:
            if condition == 1:
                distractor_ixs = np.where(valid_distractors.speaker_sex==distractors)[0]
                distractor_ixs = np.random.choice(distractor_ixs, size=1)
                noise = valid_distractors['signal'][distractor_ixs[0]]

            elif condition in [2, 4]:
                talker_sex_counts = dict(zip(*np.unique(distractors, return_counts = True)))
                m_distractors=[]
                f_distractors=[]
                if 'm' in talker_sex_counts:
                    m_distractors = np.where(valid_distractors.speaker_sex =='m')[0]
                    m_distractors = np.random.choice(m_distractors, size=talker_sex_counts['m'])
                if 'f' in talker_sex_counts:
                    f_distractors = np.where(valid_distractors.speaker_sex=='f')[0]
                    f_distractors = np.random.choice(f_distractors, size=talker_sex_counts['f'])
                distractor_ixs = np.concatenate([f_distractors, m_distractors])

                noise = np.stack(valid_distractors['signal'][distractor_ixs]).sum(0) # sum unique distractors 
            # get common feats 
            words = valid_distractors['word'][distractor_ixs].values
            indices = valid_distractors['_original_timit_index'][distractor_ixs].values
            speakers = valid_distractors['speaker'][distractor_ixs].values
        
        # create trial stim 
        noise,_ = rms_normalize(noise)
        signal, _ = rms_normalize(signal)
        mixture, first_scale_factor = combine_with_noise(signal, noise, snr)
        mixture, second_cue_scale = rms_normalize(mixture)
        cue, _ = rms_normalize(exp_df['cue_signal'][i])
        cue *= first_scale_factor
        cue *= second_cue_scale
        # rove cue
        cue_snr, cue_signal = rove_cue(cue,
                                       snr,
                                       second_cue_scale)
        cue_snrs.append(cue_snr)
        cue_signals.append(cue_signal)

        # save values for tiral 
        trial_data['mixture_signal'].append(mixture)
        trial_data['distractor_signal'].append(noise)
        trial_data['_original_distractor_timit_indices'].append(indices)
        trial_data['distractor_words'].append(words)
        trial_data['distractor_speakers'].append(speakers)
        trial_data['distractor_conditions'].append(condition)

        trial_data['distractor_sex'].append(''.join(str(token) for token in distractors))
        trial_data['snrs'].append(snr)
    # convert to trial df 
    trial_data = pd.DataFrame.from_dict(trial_data) 
    # merge with exp_df 
    exp_df = exp_df.join(trial_data)
    # update cues with roved level
    exp_df['cue_snr'] = cue_snrs
    exp_df['cue_signal'] = cue_signals
    
    # get catch trials
    catch_trials = bg_timit.sample(n=n_catch_trials)
    catch_trials.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    catch_trials.sort_values(by='speaker', inplace=True)
    catch_trials.reset_index(inplace=True, drop=True)
    
    #  catch cues are just the target signal
    catch_trials[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = catch_trials[['signal', 'word', '_original_timit_index', 'speaker']]
    
    catch_trials['mixture_signal'] = catch_trials['signal']
    catch_trials['distractor_signal'] = 'catch trial'
    catch_trials['_original_distractor_timit_indices'] = 'catch trial'
    catch_trials['distractor_words'] = 'catch trial'
    catch_trials['distractor_speakers']  = 'catch trial'
    catch_trials['distractor_conditions'] = 'catch trial'
    catch_trials['snrs'] = 'catch trial'
    
    exp_df = pd.concat([exp_df, catch_trials],ignore_index=True)

    ## Save exp_df 
    df_name = file_out_dir / f"attn_task_dataset_{set_ix:02d}.pdpkl"
#     break
    exp_df.to_pickle(df_name)
    

  0%|          | 0/10 [00:00<?, ?it/s]

215 unique words in dataset 0
6 max frequency in dataset 0



 10%|█         | 1/10 [00:04<00:44,  4.95s/it]    

217 unique words in dataset 1
6 max frequency in dataset 1



 20%|██        | 2/10 [00:09<00:37,  4.70s/it]    

...


 80%|████████  | 8/10 [00:40<00:10,  5.28s/it]    

221 unique words in dataset 8
7 max frequency in dataset 8



 90%|█████████ | 9/10 [00:45<00:05,  5.16s/it]    

221 unique words in dataset 9
7 max frequency in dataset 9



100%|██████████| 10/10 [00:50<00:00,  5.01s/it]   


In [27]:
len(cue_signals)

400

In [28]:
exp_df.head()

,_original_timit_index,word,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,...,cue_speaker,mixture_signal,distractor_signal,_original_distractor_timit_indices,distractor_words,distractor_speakers,distractor_conditions,distractor_sex,snrs,cue_snr
0,612,example,train-dr2-faem0-si1392,faem0,20000,40000,f,si,1392,dr2,...,faem0,"[-0.006822645163050587, -0.0037874132864880706...","[-0.007812780292013275, -0.004357954035195457,...","[1905, 3342]","[brought, significant]","[fcke0, fsak0]",2,ff,-6,-6.0
1,627,their,train-dr2-fajw0-si1263,fajw0,20000,40000,f,si,1263,dr2,...,fajw0,"[0.009845406383928966, -0.010773908710763768, ...","[0.007406963885266436, -0.005011124083810603, ...","[917, 4710, 9318, 223]","[cannot, received, areas, start]","[frll0, mdas0, mbpm0, mcpm0]",4,mmmf,-6,-6.0
2,7513,production,test-dr1-faks0-si943,faks0,20000,40000,f,si,943,dr1,...,faks0,"[-0.013907554790717326, -0.011303983550925888,...","[-0.019859923063286912, -0.01599053499444257, ...","[82, 9237, 204, 888]","[appointed, popular, after, capital]","[fjsp0, fjsa0, fvfb0, fpjf0]",4,ffff,0,0.0
3,3134,majority,train-dr4-falr0-si1325,falr0,20000,40000,f,si,1325,dr4,...,falr0,"[0.0060185866316945485, 0.0059010419969586754,...","[9.768441504394974e-05, 0.00011535945339393433...",[3282],[large],[fklc0],1,f,-6,-6.0
4,5345,child,train-dr6-fapb0-sx343,fapb0,20000,40000,f,sx,343,dr6,...,fapb0,"[0.004268380957657227, 0.0042787997349107345, ...","[0.004577583440843272, 0.004791912709353122, 2...","[781, 9405, 8154, 9387]","[present, working, don't, based]","[fjkl0, mdrb0, mbdg0, mdac2]",4,mmmf,-6,-6.0


In [29]:
from IPython.display import Audio

# Audio(exp_df['cue_signal'][0], rate=20000, normalize=False)

# Demo Distractor by SNR conditions 

In [30]:
isi = np.zeros(int(0.25 * 20_000))

def make_trial(cue, mixture):
    return np.hstack([cue, isi, mixture])

### 1 distractor at -8, -3, 0, and +7 dB SNR

In [31]:
one_talker_df = exp_df[exp_df['distractor_conditions'] == 1]

In [33]:
trial = one_talker_df.loc[one_talker_df['snrs'] == -6].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print("word = ", trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

word =  majority


In [ ]:
trial = one_talker_df.loc[one_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = one_talker_df.loc[one_talker_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = one_talker_df.loc[one_talker_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

### 2 distractor at -8, -3, 0, and +7 dB SNR

In [ ]:
two_talker_df = exp_df[exp_df['distractor_conditions'] == 2]

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == -8].iloc[2]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

### 4 distractor at -8, -3, 0, and +7 dB SNR

In [ ]:
four_talker_df = exp_df[exp_df['distractor_conditions'] == 4]

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == -8].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == 0].iloc[1]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

### Speech Shaped Noise distractor at -8, -3, 0, and +7 dB SNR

In [ ]:
ssn_df = exp_df[exp_df['distractor_conditions'] == 'ssn']

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == -8].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == -3].iloc[20]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

## Demo Speech Shaped noise gen

In [ ]:
def make_speech_shaped_noise(x):
    ## Use for demos 
    x = np.asarray(x)
    n_x = x.shape[-1]
    n_fft = next_pow_2(n_x)
    X = np.fft.rfft(x, next_pow_2(n_fft))
    if X.ndim == 2:
        X = X.mean(0)
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    out = noise[:n_x]

    return out

ssn = make_speech_shaped_noise(np.stack(target_timit['signal']))

In [ ]:
ssn.shape

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.abs(np.fft.rfft(ssn)))

In [ ]:
plt.plot(np.abs(np.fft.rfft(exp_df['signal'][100])))

In [ ]:
from IPython.display import Audio
Audio(ssn, rate=20000)

In [ ]:
Audio(exp_df['signal'][10], rate=20000)

In [ ]:
Audio(exp_df['mixture_signal'][4], rate=20000)